<a href="https://colab.research.google.com/github/netsetos/genai-engg-gcp-learners/blob/main/module-02-embeddings-and-vectors/lesson-2.3-firestore-vector/practice/GCP_Capstone_2.3_Practice_Lab.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Practice Lab 2.3 — Firestore Vector Search

8 exercises with complete solutions. Create indexes, store embeddings, query with find_nearest(), and build a full RAG pipeline.

Runnable companion to the published practice lab. Each exercise below shows the objective and a complete solution. Cloud Shell / `gcloud` steps are `%%bash` cells; Python steps run in Colab after you authenticate and set your project.

---

## Exercise 1: Create Vector Index  
**Difficulty:** Easy

Run the gcloud command to create a flat vector index on knowledge_base.embedding with 768 dimensions.

1. Open Cloud Shell
2. Run gcloud firestore indexes composite create
3. Verify with gcloud firestore operations list

**Solution:**

In [ ]:
# --- Setup: install + auth + project (run me first) ---
!pip install -q google-genai google-cloud-firestore
from google.colab import auth
auth.authenticate_user()

import subprocess
PROJECT_ID = 'documind-ai-YOUR-ID'   # CHANGE THIS to your project id
!gcloud config set project {PROJECT_ID}
# ensure the Firestore (default) database exists (idempotent)
if '(default)' not in subprocess.run(['gcloud','firestore','databases','list','--project',PROJECT_ID,'--format=value(name)'],capture_output=True,text=True).stdout:
    subprocess.run(['gcloud','firestore','databases','create','--location=asia-south1','--project',PROJECT_ID],check=False)
print('Setup done')

In [ ]:
%%bash
# Plain vector index (Exercise 3) -- || true so re-runs do not fail on "already exists"
gcloud firestore indexes composite create \
  --collection-group=knowledge_base --query-scope=COLLECTION \
  --field-config field-path=embedding,vector-config='{"dimension":"768","flat":"{}"}' \
  --database="(default)" || true

# Composite index category(ASC)+embedding -- needed by the FILTERED query (Exercise 5)
gcloud firestore indexes composite create \
  --collection-group=knowledge_base --query-scope=COLLECTION \
  --field-config field-path=category,order=ASCENDING \
  --field-config field-path=embedding,vector-config='{"dimension":"768","flat":"{}"}' \
  --database="(default)" || true

# Block until every index is READY (find_nearest raises FAILED_PRECONDITION until then; ~2-5 min)
echo "Waiting for vector indexes to reach READY..."
for i in $(seq 1 40); do
  STATES=$(gcloud firestore indexes composite list --database="(default)" --format='value(state)')
  if [ -n "$STATES" ] && ! echo "$STATES" | grep -q CREATING; then echo "Indexes READY"; break; fi
  sleep 15
done

## Exercise 2: Store 5 Documents with Embeddings  
**Difficulty:** Easy

Write 5 documents to Firestore, each with text content, a 768-dim embedding via Vector(), and a category field.

1. Initialize Firestore and genai clients
2. Embed 5 texts with RETRIEVAL_DOCUMENT
3. Store each with Vector(embedding)
4. Verify in Firestore Console

**Solution:**

In [ ]:
from google.cloud import firestore
from google.cloud.firestore_v1.vector import Vector
from google import genai
from google.genai import types

db = firestore.Client(project="documind-ai-YOUR-ID")
ai = genai.Client(enterprise=True, project="documind-ai-YOUR-ID", location="us-central1")
col = db.collection("knowledge_base")

texts = [
    ("ai_ml", "Neural networks learn from labeled data."),
    ("gcp", "Cloud Run deploys containers serverlessly."),
    ("india", "Hyderabad is a major tech hub in India."),
    ("python", "Python list comprehensions are concise."),
    ("ai_ml", "RAG combines retrieval with generation."),
]

# gemini-embedding-001 accepts ONE text per call on Vertex AI, so loop
for i, (cat, text) in enumerate(texts):
    emb = ai.models.embed_content(
        model="gemini-embedding-001", contents=text,
        config={"task_type":"RETRIEVAL_DOCUMENT","output_dimensionality":768}
    ).embeddings[0]
    col.document(f"test_{i}").set({
        "content": text, "embedding": Vector(emb.values), "category": cat})
print(f"Stored {len(texts)} documents")

## Exercise 3: First find_nearest() Query  
**Difficulty:** Easy

Run your first vector search. Print top-3 results with cosine similarity scores.

1. Embed a query with RETRIEVAL_QUERY
2. Call find_nearest() with COSINE distance
3. Convert distance to similarity (1 - distance)
4. Print ranked results

**Solution:**

In [ ]:
from google.cloud.firestore_v1.base_vector_query import DistanceMeasure

query = "How does deep learning work?"
q_emb = ai.models.embed_content(
    model="gemini-embedding-001", contents=query,
    config={"task_type":"RETRIEVAL_QUERY","output_dimensionality":768}
).embeddings[0].values

results = col.find_nearest(
    vector_field="embedding",
    query_vector=Vector(q_emb),
    distance_measure=DistanceMeasure.COSINE,
    limit=3,
    distance_result_field="vector_distance",
).get()

print(f"Query: {query}")
for doc in results:
    d = doc.to_dict()
    sim = 1 - d["vector_distance"]
    print(f"  [{sim:.4f}] {d['content']}")

## Exercise 4: Batch Ingest 20 Documents  
**Difficulty:** Medium

Load the full 20-document corpus using batch embeddings + batch writes. Verify all are searchable.

1. Define 20 documents across 4 categories
2. Embed all 20 in one API call
3. Batch write all 20 to Firestore
4. Run a test query to verify

**Solution:**

In [ ]:
# Self-contained mini-corpus. gemini-embedding-001 takes ONE text per call,
# so we embed in a loop and write with a single batch commit.
corpus = [
    {"content": f"{topic} is an important concept.", "category": cat}
    for topic, cat in [
        ("Neural networks", "ai_ml"), ("Cloud Run", "gcp"), ("Hyderabad", "india"),
        ("Python typing", "python"), ("Vector search", "ai_ml"), ("BigQuery", "gcp")]
]

batch = db.batch()
for i, doc_data in enumerate(corpus):
    emb = ai.models.embed_content(
        model="gemini-embedding-001", contents=doc_data["content"],
        config={"task_type":"RETRIEVAL_DOCUMENT","output_dimensionality":768}
    ).embeddings[0].values
    batch.set(col.document(f"doc_{i:03d}"), {**doc_data, "embedding": Vector(emb)})
batch.commit()
print(f"Loaded {len(corpus)} documents")

## Exercise 5: Filtered Vector Search  
**Difficulty:** Medium

Create a composite index for category+embedding. Search within "ai_ml" only. Compare with unfiltered results.

1. Create composite index via gcloud
2. Run unfiltered find_nearest()
3. Run .where("category","==","ai_ml").find_nearest()
4. Compare results and relevance

**Solution:**

In [ ]:
# Unfiltered
unfiltered = col.find_nearest(
    vector_field="embedding", query_vector=Vector(q_emb),
    distance_measure=DistanceMeasure.COSINE, limit=3,
    distance_result_field="dist").get()

# Filtered (requires composite index)
filtered = col.where("category","==","ai_ml").find_nearest(
    vector_field="embedding", query_vector=Vector(q_emb),
    distance_measure=DistanceMeasure.COSINE, limit=3,
    distance_result_field="dist").get()

print("Unfiltered:")
for d in unfiltered: print(f"  [{1-d.to_dict()['dist']:.4f}] {d.to_dict()['content'][:50]}")
print("\nFiltered (ai_ml only):")
for d in filtered: print(f"  [{1-d.to_dict()['dist']:.4f}] {d.to_dict()['content'][:50]}")

## Exercise 6: Distance Threshold Testing  
**Difficulty:** Medium

Set distance_threshold=0.3. Test with relevant and irrelevant queries. Verify threshold filters correctly.

1. Query with a relevant question (should return results)
2. Query with an irrelevant question (should return empty)
3. Compare result counts

**Solution:**

In [ ]:
for query in ["How do neural networks learn?", "What is the recipe for chocolate cake?"]:
    q = ai.models.embed_content(
        model="gemini-embedding-001", contents=query,
        config={"task_type":"RETRIEVAL_QUERY","output_dimensionality":768}
    ).embeddings[0].values
    results = col.find_nearest(
        vector_field="embedding", query_vector=Vector(q),
        distance_measure=DistanceMeasure.COSINE, limit=5,
        distance_threshold=0.3,  # similarity > 0.7
    ).get()
    print(f"Query: {query[:40]}... → {len(results)} results")

## Exercise 7: Full RAG Pipeline  
**Difficulty:** Challenge

Build rag_query() that embeds, searches Firestore, and generates with Gemini. Test across 4 categories.

1. Build function: embed → find_nearest → generate
2. Test: "How do transformers work?"
3. Test: "What is India's data protection law?" (filtered)
4. Test: "Best practices for Python async?"

**Solution:**

In [ ]:
# See Cell 6 in the Colab notebook for the complete rag_query() function
# Key: embed with RETRIEVAL_QUERY, search with COSINE, generate with thinking_budget=0

## Exercise 8: FirestoreRAG Production Module  
**Difficulty:** Challenge

Build the complete FirestoreRAG class with embed(), ingest(), search(), query() methods.

1. Define class with __init__ for clients
2. Implement embed() with configurable task type
3. Implement search() with optional category filter
4. Implement query() for full RAG
5. Test end-to-end with 20 documents

**Solution:**

In [ ]:
# See Cell 7 in the Colab notebook and Step 10 in the main lesson
# for the complete FirestoreRAG class implementation